<a href="https://colab.research.google.com/github/kmarkande/adaptive-memory-ai/blob/main/01_python_and_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# 🔴 1. Import SQLite and file-management tools
import sqlite3
import os

# 🔴 2. Define a function to check database validity
def check_database(path):

    # 🔴 3. Return False if the database file does not exist
    if not os.path.exists(path):
        return False

    try:
        connection = sqlite3.connect(
            f"file:{path}?mode=ro",
            uri=True
        )

        cursor = connection.cursor()

        # 🔴 4. Check database integrity
        cursor.execute("PRAGMA integrity_check")
        integrity = cursor.fetchone()[0]

        # 🔴 5. Check required tables
        cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        """)

        tables = {row[0] for row in cursor.fetchall()}

        connection.close()

        # 🔴 6. Confirm integrity and required tables
        return (
            integrity == "ok"
            and {"entities", "episodes"}.issubset(tables)
        )

    except sqlite3.Error:
        return False


# 🔴 7. Test the active database
if check_database(database_path):
    print("Active database is valid.")
else:
    print("Active database is missing or invalid.")

    # 🔴 1. Check whether the active database is valid
if check_database(database_path):

    print("Active database is valid. No restoration needed.")

# 🔴 2. If active database is invalid, check the Google Drive backup
elif check_database(backup_path):

    print("Active database is missing or invalid.")
    print("Restoring database from Google Drive...")

    # 🔴 3. Copy the validated backup to a temporary file
    temporary_restore = database_path + ".tmp"

    backup_connection = sqlite3.connect(
        f"file:{backup_path}?mode=ro",
        uri=True
    )

    restore_connection = sqlite3.connect(temporary_restore)

    backup_connection.backup(restore_connection)

    restore_connection.close()
    backup_connection.close()

    # 🔴 4. Validate the restored database
    if check_database(temporary_restore):

        # 🔴 5. Replace the active database
        os.replace(temporary_restore, database_path)

        print("Database restored successfully!")

    else:
        os.remove(temporary_restore)
        raise RuntimeError("Restored database failed validation.")

# 🔴 6. Stop if neither database is valid
else:

    raise RuntimeError(
        "No valid database or backup found. "
        "Database setup must be checked manually."
    )

Active database is valid.
Active database is valid. No restoration needed.
